# Speaker Embedding with pyannote and OpenVINO

Speaker embedding turns a speech recording into a fixed-size vector that captures *who* is speaking, independent of *what* is said. Two clips from the same speaker produce vectors with a small cosine distance, while clips from different speakers are far apart. These embeddings are the core building block of speaker verification, speaker comparison, and speaker diarization systems.

This notebook uses the open-source [`pyannote/embedding`](https://huggingface.co/pyannote/embedding) model and accelerates it with OpenVINO. The model is an x-vector TDNN speaker embedding network with a trainable SincNet front-end that maps a 16 kHz mono waveform to a 512-dimensional speaker vector. Its model card reports a 2.8% Equal Error Rate (EER) on the VoxCeleb1 test set using cosine distance directly, without VAD or PLDA post-processing.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Hugging Face access](#Hugging-Face-access)
- [Load the embedding model](#Load-the-embedding-model)
- [Prepare sample audio](#Prepare-sample-audio)
- [Extract embeddings with PyTorch](#Extract-embeddings-with-PyTorch)
- [Convert the model to OpenVINO](#Convert-the-model-to-OpenVINO)
    - [Convert to OpenVINO IR](#Convert-to-OpenVINO-IR)
    - [Select inference device](#Select-inference-device)
    - [Extract embeddings with OpenVINO](#Extract-embeddings-with-OpenVINO)
- [Optional: run on Intel XPU](#Optional:-run-on-Intel-XPU)
- [Optional: VoxCeleb1 EER benchmark](#Optional:-VoxCeleb1-EER-benchmark)
- [Cleanup](#Cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/pyannote-audio/pyannote-embedding.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install the required packages. `pyannote.audio` provides the embedding model, OpenVINO accelerates it, and `datasets` / `scikit-learn` are used for the sample audio and the optional EER benchmark.


In [ ]:
import platform

%pip install -q "openvino>=2025.1.0"
%pip install -q "jupyter" "ipykernel"
%pip install -q "pyannote.audio>=4.0.0" "omegaconf" "onnx" "soundfile" "librosa" "scipy" "scikit-learn" "datasets" "ipywidgets" "torch>=2.4.0" "torchaudio" --extra-index-url https://download.pytorch.org/whl/cpu

import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("pyannote-embedding.ipynb")


## Hugging Face access
[back to top ⬆️](#Table-of-contents:)

`pyannote/embedding` is a **gated** model. Before running this notebook you need to:

1. Create a *Read* access token at https://huggingface.co/settings/tokens.
2. Accept the user conditions at https://huggingface.co/pyannote/embedding.

Then log in with your token. The token is stored locally by `huggingface_hub` and reused on later runs.


In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()
else:
    print("Already logged in to Hugging Face.")


## Load the embedding model
[back to top ⬆️](#Table-of-contents:)

`Model.from_pretrained` downloads the model checkpoint (cached after the first run). The model runs on CPU PyTorch here; it is moved to OpenVINO later.


In [ ]:
import torch
from pyannote.audio import Model

MODEL_ID = "pyannote/embedding"
SAMPLE_RATE = 16000  # the model is trained on 16 kHz audio

model = Model.from_pretrained(MODEL_ID)
if model is None:
    raise RuntimeError(
        f"Could not load '{MODEL_ID}'. Accept the model conditions on Hugging Face "
        "and make sure you are logged in (see the previous cell)."
    )
model = model.eval().to(torch.device("cpu"))
print("Model loaded.")


## Prepare sample audio
[back to top ⬆️](#Table-of-contents:)

We stream a few short clips from the [LibriSpeech](https://huggingface.co/datasets/openslr/librispeech_asr) `dev-clean` set (16 kHz, mono). Shuffling the stream lets us grab two clips from one speaker (**A-1**, **A-2**) and one clip from a different speaker (**B-1**) after only a handful of samples, without downloading the whole dataset. We later verify that same-speaker embeddings are close and different-speaker embeddings are far apart.


In [ ]:
from collections import defaultdict

import numpy as np
from datasets import load_dataset

# LibriSpeech dev-clean has 40 speakers; shuffle the stream so speakers are mixed
# and we can grab two clips from one speaker and one from another after just a few
# samples, without downloading the whole dataset.
stream = load_dataset("openslr/librispeech_asr", "clean", split="validation", streaming=True).shuffle(seed=0, buffer_size=300)

by_speaker = defaultdict(list)
for item in stream:
    by_speaker[item["speaker_id"]].append(np.asarray(item["audio"]["array"], dtype="float32"))
    speakers_with_two = [s for s, clips in by_speaker.items() if len(clips) >= 2]
    other_speakers = [s for s in by_speaker if s not in speakers_with_two]
    if speakers_with_two and other_speakers:
        spk_a, spk_b = speakers_with_two[0], other_speakers[0]
        break

samples = {
    "A-1": by_speaker[spk_a][0],
    "A-2": by_speaker[spk_a][1],
    "B-1": by_speaker[spk_b][0],
}

print(f"Speaker A = {spk_a}, Speaker B = {spk_b}")
for name, waveform in samples.items():
    print(f"  {name}: {waveform.shape[0] / SAMPLE_RATE:.1f}s")


## Extract embeddings with PyTorch
[back to top ⬆️](#Table-of-contents:)

Run the model as-is on PyTorch to get baseline embeddings, then compare speakers with cosine distance (0 = identical, larger = more different).


In [ ]:
def to_model_input(waveform: np.ndarray) -> np.ndarray:
    """Reshape a 1-D waveform to the model input shape (1, 1, samples)."""
    return np.asarray(waveform, dtype="float32").reshape(1, 1, -1)


def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine distance between two embedding vectors (0 = identical)."""
    a = np.asarray(a, dtype="float64").reshape(-1)
    b = np.asarray(b, dtype="float64").reshape(-1)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 1.0 if denom == 0 else float(1.0 - np.dot(a, b) / denom)


def embed_torch(waveform: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.from_numpy(to_model_input(waveform))).cpu().numpy()[0]


def report(embeddings: dict) -> None:
    """Print the same-speaker and different-speaker cosine distances."""
    same = cosine_distance(embeddings["A-1"], embeddings["A-2"])
    diff = cosine_distance(embeddings["A-1"], embeddings["B-1"])
    print(f"cosine distance A-1 vs A-2 (same speaker):      {same:.3f}")
    print(f"cosine distance A-1 vs B-1 (different speakers): {diff:.3f}")


torch_emb = {name: embed_torch(w) for name, w in samples.items()}
for name, emb in torch_emb.items():
    print(f"{name}: shape={emb.shape} norm={np.linalg.norm(emb):.3f}")

print()
report(torch_emb)


## Convert the model to OpenVINO
[back to top ⬆️](#Table-of-contents:)

The embedding model is a single neural block that maps a waveform to a 512-dimensional vector. We convert it to OpenVINO IR and run inference through the OpenVINO runtime.


### Convert to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Trace the model and save it as OpenVINO IR (`.xml` / `.bin`) into `ov_models/`, keeping the time axis dynamic so the IR accepts any waveform length. IR weights are stored in FP16 by default (`ov.save_model(..., compress_to_fp16=True)`).


In [ ]:
from unittest.mock import MagicMock

import openvino as ov

OV_MODEL_DIR = Path("ov_models")
OV_MODEL_DIR.mkdir(exist_ok=True)
EMBEDDING_XML = OV_MODEL_DIR / "pyannote_embedding.xml"


class EmbeddingWrapper(torch.nn.Module):
    """Expose a single clean forward for tracing."""

    def __init__(self, embedding_model: Model) -> None:
        super().__init__()
        self.model = embedding_model

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        return self.model(waveform)


if not EMBEDDING_XML.exists():
    # pyannote models are Lightning modules; a dummy trainer lets TorchScript trace them.
    for module in model.modules():
        module._trainer = MagicMock()
    wrapped = EmbeddingWrapper(model).eval()
    example = torch.randn(1, 1, SAMPLE_RATE * 2)
    with torch.no_grad():
        traced = torch.jit.trace(wrapped, example, strict=False)
    ov_model = ov.convert_model(traced, example_input=example, input=[ov.PartialShape([1, 1, -1])])
    ov.save_model(ov_model, EMBEDDING_XML)
print(f"OpenVINO IR is ready at '{EMBEDDING_XML}'.")


### Select inference device
[back to top ⬆️](#Table-of-contents:)

Select the OpenVINO device for inference from the dropdown. Choose `CPU`, `GPU` (if you have an Intel GPU), or `AUTO`.


In [ ]:
import openvino as ov

from notebook_utils import device_widget

core = ov.Core()
print("Available OpenVINO devices:")
for d in core.available_devices:
    print(f"  {d}: {core.get_property(d, 'FULL_DEVICE_NAME')}")

device = device_widget(default="AUTO", exclude=["NPU"])
device


### Extract embeddings with OpenVINO
[back to top ⬆️](#Table-of-contents:)

Run the same clips through the OpenVINO IR and compare the results with the PyTorch baseline. Intel GPUs are slow with dynamic shapes, so on GPU the IR is reshaped to each clip's exact length (a static shape, compiled once per length); on CPU the dynamic IR is used directly.


In [ ]:
class OVEmbedder:
    """Extract embeddings through the OpenVINO IR.

    On GPU a static input shape is required, so we compile one model per unique
    clip length; on CPU the dynamic IR is reused for every call.
    """

    def __init__(self, core: ov.Core, xml: str, device: str, static: bool) -> None:
        self.core, self.xml, self.device, self.static = core, xml, device, static
        self._by_len: dict = {}
        self._dynamic = None if static else core.compile_model(core.read_model(xml), device)

    def __call__(self, waveform: np.ndarray) -> np.ndarray:
        model_input = to_model_input(waveform)
        if self.static:
            num_samples = model_input.shape[-1]
            compiled = self._by_len.get(num_samples)
            if compiled is None:
                ov_model = self.core.read_model(self.xml)
                ov_model.reshape({0: ov.PartialShape([1, 1, num_samples])})
                compiled = self.core.compile_model(ov_model, self.device)
                self._by_len[num_samples] = compiled
        else:
            compiled = self._dynamic
        return compiled(model_input)[compiled.output(0)][0]


ov_device = device.value
static = "GPU" in ov_device or (ov_device == "AUTO" and any("GPU" in d for d in core.available_devices))
embedder = OVEmbedder(core, str(EMBEDDING_XML), ov_device, static)
print(f"Extracting embeddings on OpenVINO device: {ov_device} (static shapes: {static})\n")

ov_emb = {name: embedder(w) for name, w in samples.items()}
report(ov_emb)

print("\nPyTorch vs OpenVINO cosine distance per clip (should be ~0):")
for name in samples:
    print(f"  {name}: {cosine_distance(torch_emb[name], ov_emb[name]):.5f}")


## Optional: run on Intel XPU
[back to top ⬆️](#Table-of-contents:)

As an alternative to OpenVINO, the model can also run on an Intel GPU through PyTorch's **XPU** backend with `model.to("xpu")`.

This section is **optional** and only runs when an Intel XPU device is detected (it needs the XPU build of PyTorch). On machines without an XPU — including CI — it is skipped automatically.

To enable it on a Linux machine with an Intel GPU, set `INSTALL_XPU = True` in the next cell to install the XPU build of PyTorch, **restart the kernel**, then re-run the notebook.


In [ ]:
# The XPU backend needs the Intel XPU build of PyTorch (Linux + Intel GPU only).
# It replaces the CPU torch/torchaudio in this kernel, so RESTART THE KERNEL after
# installing and then re-run the notebook. Leave this False on CPU-only machines
# and in CI.
INSTALL_XPU = False

import platform
import subprocess
import sys

import torch

if "+xpu" in torch.__version__:
    print(f"XPU build already installed: torch {torch.__version__} - nothing to do.")
elif INSTALL_XPU and platform.system() == "Linux":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--force-reinstall", "torch", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/xpu"]
    )
    print("\nXPU PyTorch installed. Please RESTART THE KERNEL, then run the notebook again.")
else:
    print("XPU install skipped. Set INSTALL_XPU = True on a Linux machine with an Intel GPU to install it.")


### Run embeddings on the Intel XPU
[back to top ⬆️](#Table-of-contents:)

With the XPU build of PyTorch installed and the kernel restarted, extract the sample embeddings on the Intel GPU. The cell auto-skips when no XPU device is present.


In [ ]:
RUN_XPU = True  # only runs when an Intel XPU device is actually present

if RUN_XPU and hasattr(torch, "xpu") and torch.xpu.is_available():
    print(f"Intel XPU detected: {torch.xpu.get_device_name(0)}\n")
    try:
        xpu_model = Model.from_pretrained(MODEL_ID).eval().to(torch.device("xpu"))

        def embed_xpu(waveform: np.ndarray) -> np.ndarray:
            with torch.no_grad():
                tensor = torch.from_numpy(to_model_input(waveform)).to("xpu")
                return xpu_model(tensor).cpu().numpy()[0]

        xpu_emb = {name: embed_xpu(w) for name, w in samples.items()}
        report(xpu_emb)
    except Exception as error:
        print(f"XPU run failed: {error}")
elif RUN_XPU:
    print("No Intel XPU available - skipping. Install the XPU build of PyTorch and run on an Intel GPU to enable it.")
else:
    print("XPU run disabled. Set RUN_XPU = True to try it.")


## Optional: VoxCeleb1 EER benchmark
[back to top ⬆️](#Table-of-contents:)

The cells below reproduce the speaker-verification **Equal Error Rate (EER)** on the [VoxCeleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html) test set. Every unique clip in the trial list is embedded once, each trial pair is scored by cosine similarity, and the EER is computed from the ROC curve.

This is **disabled by default** because it downloads about 1 GB of audio and takes much longer than the rest of the notebook. Set `RUN_EER_BENCHMARK = True` in the next cell to automatically download the trial list and test audio (cross-platform, pure Python) and run the benchmark. The model card reports **2.8%** EER on the full VoxCeleb1 test set.


In [ ]:
RUN_EER_BENCHMARK = False  # set True to download VoxCeleb1 test data and run the EER benchmark
VOX_ROOT = Path("voxceleb1_test")  # dataset root populated by the download step below

if RUN_EER_BENCHMARK:
    import zipfile

    from huggingface_hub import hf_hub_download

    VOX_ROOT.mkdir(exist_ok=True)
    trials_path = VOX_ROOT / "veri_test.txt"
    wav_root = VOX_ROOT / "vox1" / "wav"

    # 1. Trial pairs list (public, ~1.5 MB, 37,720 pairs).
    if not trials_path.exists():
        print("Downloading VoxCeleb1 trial list...")
        trials_path.write_bytes(
            requests.get("https://mm.kaist.ac.kr/datasets/voxceleb/meta/veri_test.txt", timeout=60).content
        )
    print(f"Trial pairs: {sum(1 for _ in open(trials_path))}")

    # 2. Test audio (~1 GB) from a Hugging Face mirror of VoxCeleb1.
    if not (wav_root.exists() and any(wav_root.rglob("*.wav"))):
        print("Downloading VoxCeleb1 test audio (~1 GB, this can take a while)...")
        zip_path = hf_hub_download(
            "ProgramComputer/voxceleb", "vox1/vox1_test_wav.zip", repo_type="dataset", local_dir=str(VOX_ROOT)
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(VOX_ROOT / "vox1")
    print(f"WAV files: {len(list(wav_root.rglob('*.wav')))}")
    print("VoxCeleb1 test set ready.")
else:
    print("EER benchmark disabled. Set RUN_EER_BENCHMARK = True to download and run it.")


### Score the EER
[back to top ⬆️](#Table-of-contents:)

Embed every unique clip with the OpenVINO IR, score each trial pair by cosine similarity, and compute the Equal Error Rate from the ROC curve. The clips have thousands of different lengths, so a per-length static GPU model is infeasible; the benchmark embeds on **CPU with a single dynamic-shape model** (one compile, exact clip lengths).


In [ ]:
if RUN_EER_BENCHMARK:
    import soundfile as sf
    from sklearn.metrics import roc_curve


    def load_audio_file(path: str) -> np.ndarray:
        """Load a wav as a mono float32 waveform resampled to 16 kHz."""
        waveform, sr = sf.read(path, dtype="float32", always_2d=True)
        waveform = waveform.mean(axis=1)
        if sr != SAMPLE_RATE:
            from math import gcd

            from scipy.signal import resample_poly

            g = gcd(int(sr), SAMPLE_RATE)
            waveform = resample_poly(waveform, SAMPLE_RATE // g, sr // g).astype("float32")
        return np.ascontiguousarray(waveform, dtype="float32")


    trials = []
    with open(trials_path) as handle:
        for line in handle:
            parts = line.split()
            if len(parts) == 3:
                trials.append((int(parts[0]), parts[1], parts[2]))
    unique_clips = sorted({p for _, a, b in trials for p in (a, b)})
    print(f"{len(trials)} pairs, {len(unique_clips)} unique clips")

    # The benchmark embeds thousands of variable-length clips. Compiling a separate
    # static GPU model per length is infeasible, so we embed on CPU with a single
    # dynamic-shape model - one compile, exact clip lengths, faithful EER.
    bench_model = core.compile_model(core.read_model(str(EMBEDDING_XML)), "CPU")
    bench_out = bench_model.output(0)

    embeddings = {}
    for i, rel in enumerate(unique_clips, 1):
        model_input = to_model_input(load_audio_file(str(wav_root / rel)))
        embeddings[rel] = bench_model(model_input)[bench_out][0]
        if i % 500 == 0:
            print(f"  embedded {i}/{len(unique_clips)}", flush=True)

    def unit(vec: np.ndarray) -> np.ndarray:
        return vec / np.linalg.norm(vec)

    scores = np.array([float(np.dot(unit(embeddings[a]), unit(embeddings[b]))) for _, a, b in trials])
    labels = np.array([label for label, _, _ in trials])

    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    eer = (fpr[idx] + fnr[idx]) / 2.0 * 100.0
    print(f"\nEER = {eer:.2f}%  (threshold cos-sim = {thresholds[idx]:.4f})")
    print("(model card reports 2.8% on the full VoxCeleb1 test set)")
else:
    print("EER benchmark skipped. Set RUN_EER_BENCHMARK = True to run it.")


## Cleanup
[back to top ⬆️](#Table-of-contents:)

Uncomment the lines below to remove the exported OpenVINO IR and the downloaded VoxCeleb1 data.


In [ ]:
import shutil

# shutil.rmtree(OV_MODEL_DIR, ignore_errors=True)
# shutil.rmtree(VOX_ROOT, ignore_errors=True)
# Path("notebook_utils.py").unlink(missing_ok=True)
